# ⚽ Fotbolls-videoanalys med YOLOv8 – Google Colab

Kör cellerna uppifrån och ner. **Rekommenderat:** Välj `Runtime → Change runtime type → GPU` för snabbare analys.

Denna notebook:
1. Installerar beroenden
2. Låter dig ladda upp en video eller länka från YouTube
3. Kör detektering + spårning med YOLOv8
4. Visar heatmaps, possession och insikter

## 1. Installera beroenden

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml tqdm yt-dlp seaborn scipy

## 2. Hämta projektkoden

Klonar repot in i Colab-sessionen. Är repot **privat** behöver du autentisera – enklast via en Personal Access Token, t.ex.:

```python
!git clone https://<DIN_TOKEN>@github.com/ehoukhe/football-video-analysis.git /content/football-video-analysis
```

Är repot publikt fungerar cellen nedan direkt.

In [ ]:
import os, sys

# Klona projektet in i Colab-sessionen (byt ev. till token-URL för privat repo).
if not os.path.isdir('/content/football-video-analysis'):
    !git clone https://github.com/ehoukhe/football-video-analysis.git /content/football-video-analysis

os.chdir('/content/football-video-analysis')
sys.path.insert(0, os.getcwd())
print('Arbetskatalog:', os.getcwd())

## 3. Välj video

Ladda upp en fil eller ange en YouTube-URL.

In [ ]:
from src.video import download_youtube

YOUTUBE_URL = ''  # klistra in en URL här, eller lämna tomt för uppladdning

if YOUTUBE_URL:
    video_path = download_youtube(YOUTUBE_URL, out_dir='data')
else:
    from google.colab import files
    uploaded = files.upload()
    os.makedirs('data', exist_ok=True)
    name = next(iter(uploaded))
    video_path = 'data/' + name
    os.replace(name, video_path)

print('Video:', video_path)

## 4. Kör analysen

Vi kör ett snabbtest på första 500 rutorna. Öka `max_frames` till 0 för hela matchen.

In [ ]:
from src.pipeline import analyze_video, load_config

config = load_config('config/config.yaml')
config['video']['max_frames'] = 500      # snabbtest; 0 = hela videon
config['model']['device'] = '0'          # använd GPU

result = analyze_video(video_path, config)
print('Klar!')

## 5. Resultat: statistik och insikter

In [ ]:
import json
print(json.dumps(result.stats.summary(), ensure_ascii=False, indent=2))
print('\n=== Insikter till tränaren ===')
for i, ins in enumerate(result.insights, 1):
    print(f'{i}. {ins}')

## 6. Visa heatmaps

In [ ]:
from IPython.display import Image, display
for hm in result.heatmaps:
    display(Image(filename=hm))

## 7. (Valfritt) Ladda ner den annoterade videon

In [ ]:
from google.colab import files
if result.output_video:
    files.download(result.output_video)